In [ ]:
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from Bio import Phylo

OUT = Path("results/phylogeny")
OUT.mkdir(parents=True, exist_ok=True)
tree_path = OUT / "colap_ml.treefile"
log_path = OUT / "colap_ml.iqtree"
if not tree_path.exists():
    iqtree = shutil.which("iqtree2") or shutil.which("iqtree")
    if iqtree is None:
        raise SystemExit("IQ-TREE 2 is not installed and results/phylogeny/colap_ml.treefile is absent")
    subprocess.run([
        iqtree, "-s", "results/alignment/colap_trimmed_aligned.fasta",
        "-m", "MFP", "-B", "1000", "-alrt", "1000", "-bnni", "--keep-ident",
        "-seed", "12345", "-T", "4", "--prefix", str(OUT / "colap_ml"),
    ], check=True)
tree = Phylo.read(tree_path, "newick")


def support_of(clade):
    text = str(clade.name) if clade.name else ""
    if "/" not in text:
        return None
    left, right = text.split("/", 1)
    try:
        return float(left), float(right)
    except ValueError:
        return None


supported = 0
ufboot95 = 0
shalrt80 = 0
internal = 0
for clade in tree.get_nonterminals():
    pair = support_of(clade)
    if pair is None:
        continue
    sh, uf = pair
    internal += 1
    shalrt80 += int(sh >= 80)
    ufboot95 += int(uf >= 95)
    supported += int(sh >= 80 and uf >= 95)
model = ""
log_likelihood = ""
bic = ""
if log_path.exists():
    for line in log_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("Best-fit model according to BIC:"):
            model = line.split(":", 1)[1].strip()
        elif line.startswith("Log-likelihood of the tree:"):
            log_likelihood = line.split(":", 1)[1].split()[0]
        elif line.startswith("Bayesian information criterion (BIC) score:"):
            bic = line.split(":", 1)[1].strip()
pd.DataFrame([{
    "tips": len(tree.get_terminals()),
    "internal_branches_with_support": internal,
    "sh_alrt_at_least_80": shalrt80,
    "ufboot_at_least_95": ufboot95,
    "both_thresholds": supported,
    "model_bic": model,
    "log_likelihood": log_likelihood,
    "bic": bic,
    "seed": 12345,
    "ufboot": 1000,
    "sh_alrt": 1000,
}]).to_csv(OUT / "tree_support.csv", index=False)
